In [ ]:
import os
    
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Rectangle
from matplotlib import animation
from IPython.display import HTML
import seaborn as sns
from PIL import Image

from tqdm import tqdm
from scipy.spatial.distance import pdist, cdist, squareform
from scipy.stats import pearsonr, spearmanr, entropy, rankdata
from sklearn.manifold import MDS
from scipy.ndimage import gaussian_filter1d

from numpy.linalg import inv, eig, pinv, solve
from scipy.linalg import svd, svdvals
from numpy import dot, multiply, diag, power
from numpy import pi, exp, sin, cos
from numpy.linalg import inv, eig, pinv, solve
from scipy.linalg import svd, svdvals
from math import floor, ceil # python 3.x

import manifold_dynamics.tuning_utils as tut

RAND = 0
RESP = (50,220)
BASE = (-50,0)
ONSET = 50
RESP = slice(ONSET + RESP[0], ONSET + RESP[1])
BASE = slice(ONSET + BASE[0], ONSET + BASE[1])

In [ ]:
DATA_DIR = '../../datasets/NNN/'
IMG_DIR = '../../datasets/NNN/NSD1000_LOC'

CAT = 'face'
dat = pd.read_pickle(os.path.join(DATA_DIR, (f'{CAT}_roi_data.pkl')))
ROI_LIST = list(dat['roi'].unique())
print(f'Unique face ROIs: {ROI_LIST}')

with open(f'../../datasets/NNN/{CAT}_mins.pkl', 'rb') as f:
    mins = pickle.load(f)
    
SCALES = {k: v[0] for k,v in mins.items()}

SAVE_DIR = '../../../buckets/manifold-dynamics/time-time/'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

In [ ]:
ROI = 'MF1_9_F' # Unknown_19_F, MF1_7_F, MF1_8_F, MF1_9_F, AF3_18_F, MB1_3_B
MODE = 'top'
scale = SCALES[ROI]

X = tut.trial_averaged_psth(dat, ROI)  # (units, time, images)
order = tut.rank_images_by_response(X)      # IMPORTANT: rank on the original X (or on split A if CV)
scale = SCALES[ROI]
idx = order[:scale]
print(f'All-image data shape: {X.shape}')

In [ ]:
idx = order[:scale]
R, rdv = tut.tuning_rdm(X, idx)

D = rdv.T
# DMD inputs: consecutive time snapshots
Xi_ = D[:, :-1]
Yi_ = D[:,  1:]

# rank selection via SVHT
sv = svdvals(Xi_)
tau = svht(Xi_, sv=sv)
r = int(np.sum(sv > tau))

lam, Phi, _ = dmd2(Xi_, Yi_, r=20)

# --- plot eigenvalues wrt unit circle ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# unit circle (quarter shown in your original plot; use full if you prefer)
p = np.linspace(0, np.pi/2, 200)
axes[0].plot(np.cos(p), np.sin(p), c="k")
axes[0].scatter(lam.real, lam.imag, color="black")
axes[0].set_aspect("equal")
axes[0].set_title("wrt unit circle")

axes[1].scatter(lam.real, lam.imag, color="black")
axes[1].set_title(f"DMD on RDV(t) | scale={scale}")

plt.show()

print(np.mean(np.abs(lam)))
print(np.mean(np.real(lam)))
print(np.sum(np.abs(lam) > 0.7))

In [ ]:
idx = np.random.choice(order, scale)
R, rdv = tut.tuning_rdm(X, idx)

D = rdv.T
# DMD inputs: consecutive time snapshots
Xi_ = D[:, :-1]
Yi_ = D[:,  1:]

# rank selection via SVHT
sv = svdvals(Xi_)
tau = svht(Xi_, sv=sv)
r = int(np.sum(sv > tau))

lam, Phi, _ = dmd2(Xi_, Yi_, r=20)

# --- plot eigenvalues wrt unit circle ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# unit circle (quarter shown in your original plot; use full if you prefer)
p = np.linspace(0, np.pi/2, 200)
axes[0].plot(np.cos(p), np.sin(p), c="k")
axes[0].scatter(lam.real, lam.imag, color="black")
axes[0].set_aspect("equal")
axes[0].set_title("wrt unit circle")

axes[1].scatter(lam.real, lam.imag, color="black")
axes[1].set_title(f"DMD on RDV(t) | scale={scale}")

plt.show()

print(np.mean(np.abs(lam)))
print(np.mean(np.real(lam)))
print(np.sum(np.abs(lam) > 0.7))

In [ ]:
Xk = X[:, :, idx]
print(f'Top-k data shape: {Xk.shape}')

# center the data per unit and per image
X_centered = Xk - Xk.mean(axis=1, keepdims=True)
print('Centered shape:', X_centered.shape)

Xi = np.mean(X_centered, axis=2)
Xi_ = Xi[:, :-1]
Yi_ = Xi[:,  1:]

fig,ax = plt.subplots(1,1)
sns.scatterplot(x=range(len(sv)), y=sv, ax=ax)
ax.axhline(tau, color='red', linestyle='dashed', linewidth=0.75)
ax.set_title(f'Optimal rank for rank reduction: r={r}')
plt.show()

In [ ]:
def dmd(X, Y, truncate=None):
    '''
    version created by Robert Taylor
    for more info: https://humaticlabs.com/blog/dmd-python/

    in this funciton, truncate == r
    '''
    if truncate == 0:
        # return empty vectors
        mu = np.array([], dtype='complex')
        Phi = np.zeros([X.shape[0], 0], dtype='complex')
    else:
        U2,Sig2,Vh2 = svd(X, False) # SVD of input matrix
        r = len(Sig2) if truncate is None else truncate # rank truncation
        U = U2[:,:r]
        Sig = diag(Sig2)[:r,:r]
        V = Vh2.conj().T[:,:r]
        Atil = dot(dot(dot(U.conj().T, Y), V), inv(Sig)) # build A tilde
        mu,W = eig(Atil)
        Phi = dot(dot(dot(Y, V), inv(Sig)), W) # build DMD modes
    return mu, Phi

def svht(X, sv=None):
    # svht for sigma unknown
    m,n = sorted(X.shape) # ensures m <= n
    beta = m / n # ratio between 0 and 1
    if sv is None:
        sv = svdvals(X)
    sv = np.squeeze(sv)
    omega_approx = 0.56 * beta**3 - 0.95 * beta**2 + 1.82 * beta + 1.43
    return np.median(sv) * omega_approx

def mrdmd(D, level=0, bin_num=0, offset=0, max_levels=7, max_cycles=2, do_svht=True):
    """
    Compute the multi-resolution DMD on the dataset `D`, returning a list of nodes
    in the hierarchy. Each node represents a particular "time bin" (window in time) at
    a particular "level" of the recursion (time scale). The node is an object consisting
    of the various data structures generated by the DMD at its corresponding level and
    time bin. The `level`, `bin_num`, and `offset` parameters are for record keeping 
    during the recursion and should not be modified unless you know what you are doing.
    The `max_levels` parameter controls the maximum number of levels. The `max_cycles`
    parameter controls the maximum number of mode oscillations in any given time scale 
    that qualify as "slow". The `do_svht` parameter indicates whether or not to perform
    optimal singular value hard thresholding.

    More info here: https://humaticlabs.com/blog/mrdmd-python/
    """

    # 4 times nyquist limit to capture cycles
    nyq = 8 * max_cycles

    # time bin size
    bin_size = D.shape[1]
    if bin_size < nyq:
        return []

    # extract subsamples 
    step = floor(bin_size / nyq) # max step size to capture cycles
    _D = D[:,::step]
    X = _D[:,:-1]
    Y = _D[:,1:]

    # determine rank-reduction
    if do_svht:
        _sv = svdvals(_D)
        tau = svht(_D, sv=_sv)
        r = sum(_sv > tau)
    else:
        r = min(X.shape)

    # compute dmd
    mu,Phi = dmd(X, Y, r)

    # frequency cutoff (oscillations per timestep)
    rho = max_cycles / bin_size

    # consolidate slow eigenvalues (as boolean mask)
    slow = (np.abs(np.log(mu) / (2 * pi * step))) <= rho
    n = sum(slow) # number of slow modes

    # extract slow modes (perhaps empty)
    mu = mu[slow]
    Phi = Phi[:,slow]

    if n > 0:

        # vars for the objective function for D (before subsampling)
        Vand = np.vander(power(mu, 1/step), bin_size, True)
        P = multiply(dot(Phi.conj().T, Phi), np.conj(dot(Vand, Vand.conj().T)))
        q = np.conj(diag(dot(dot(Vand, D.conj().T), Phi)))

        # find optimal b solution
        b_opt = solve(P, q).squeeze()

        # time evolution
        Psi = (Vand.T * b_opt).T

    else:

        # zero time evolution
        b_opt = np.array([], dtype='complex')
        Psi = np.zeros([0, bin_size], dtype='complex')

    # dmd reconstruction
    D_dmd = dot(Phi, Psi)   

    # remove influence of slow modes
    D = D - D_dmd

    # record keeping
    node = type('Node', (object,), {})()
    node.level = level            # level of recursion
    node.bin_num = bin_num        # time bin number
    node.bin_size = bin_size      # time bin size
    node.start = offset           # starting index
    node.stop = offset + bin_size # stopping index
    node.step = step              # step size
    node.rho = rho                # frequency cutoff
    node.r = r                    # rank-reduction
    node.n = n                    # number of extracted modes
    node.mu = mu                  # extracted eigenvalues
    node.Phi = Phi                # extracted DMD modes
    node.Psi = Psi                # extracted time evolution
    node.b_opt = b_opt            # extracted optimal b vector
    nodes = [node]

    # split data into two and do recursion
    if level < max_levels:
        split = ceil(bin_size / 2) # where to split
        nodes += mrdmd(
            D[:,:split],
            level=level+1,
            bin_num=2*bin_num,
            offset=offset,
            max_levels=max_levels,
            max_cycles=max_cycles,
            do_svht=do_svht
            )
        nodes += mrdmd(
            D[:,split:],
            level=level+1,
            bin_num=2*bin_num+1,
            offset=offset+split,
            max_levels=max_levels,
            max_cycles=max_cycles,
            do_svht=do_svht
            )

    return nodes

def stitch(nodes, level):
    
    # get length of time dimension
    start = min([nd.start for nd in nodes])
    stop = max([nd.stop for nd in nodes])
    t = stop - start

    # extract relevant nodes
    nodes = [n for n in nodes if n.level == level]
    nodes = sorted(nodes, key=lambda n: n.bin_num)
    
    # stack DMD modes
    Phi = np.hstack([n.Phi for n in nodes])
    
    # allocate zero matrix for time evolution
    nmodes = sum([n.n for n in nodes])
    Psi = np.zeros([nmodes, t], dtype='complex')
    
    # copy over time evolution for each time bin
    i = 0
    for n in nodes:
        _nmodes = n.Psi.shape[0]
        Psi[i:i+_nmodes,n.start:n.stop] = n.Psi
        i += _nmodes
    
    return Phi,Psi

def dmd2(X_, Y_, r):
    # X: (units, time)
    # X_ = X[:, :-1]
    # Y_ = X[:,  1:]
    U,S,Vt = np.linalg.svd(X_, full_matrices=False)
    U_r, S_r, V_r = U[:,:r], S[:r], Vt[:r,:].T
    Atilde = (U_r.T @ Y_) @ (V_r * (1.0/S_r))
    lam, W = np.linalg.eig(Atilde)
    Phi = Y_ @ (V_r * (1.0/S_r)) @ W
    return lam, Phi, (U_r,S_r,V_r,Atilde)

def delay_embed(Z, L):
    # Z: (k, T), returns (k*L, T-L+1) stacked windows within an image
    T = Z.shape[1]
    return np.vstack([Z[:, s:T-L+s+1] for s in range(L)])

def mode_amplitudes(Phi, W_img):
    # initial coefficients for image i (at the first embedded time)
    return np.linalg.lstsq(Phi, W_img[:, 0], rcond=None)[0]